# LawNet Case Scraper

This notebook scrapes case data from LawNet OpenLaw. It uses **Playwright** to handle the dynamic JavaScript content of the website.

## Setup
First, we need to install the necessary libraries.

In [ ]:
%pip install playwright nest_asyncio beautifulsoup4 pandas
!playwright install chromium

  Using cached pyee-13.0.0-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/36.8 MB ? eta -:--:--
   - -------------------------------------- 1.8/36.8 MB 20.2 MB/s eta 0:00:02
   ------ --------------------------------- 6.3/36.8 MB 21.5 MB/s eta 0:00:02
   ----------- ---------------------------- 10.5/36.8 MB 20.4 MB/s eta 0:00:02
   --------------- ------------------------ 14.7/36.8 MB 20.1 MB/s eta 0:00:02
   -------------------- ------------------- 19.1/36.8 MB 20.5 MB/s eta 0:00:01
   -------------------------- ------------- 24.1/36.8 MB 21.2 MB/s eta 0:00:01
   -------------------------------- ------- 29.9/36.8 MB 21.8 MB/s eta 0:00:01
   ------------------------------------- -- 34.3/36.8 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------  36.7/36.8 MB 21.4 MB/s eta 0:00:01
   ---------------------------------------- 36.8/36.8 MB 18.7 MB/s  0:00:02
Using cached pyee-13.0.0-py3-none-any.whl (15 kB)

   ------------- ---------------

## Imports and Configuration

In [ ]:
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import re

# Apply nest_asyncio to allow Playwright to run in Jupyter
nest_asyncio.apply()

## Scraping Function

We define a function `scrape_lawnet_case` that:
1. Launches a headless browser.
2. Navigates to the case URL.
3. Waits for the content to load.
4. Extracts key information (Title, Citation, Date, Coram, Text).

> **Note:** Since LawNet is a dynamic app, we wait for the network to be idle and for specific text to appear.

In [ ]:
async def scrape_lawnet_case(url):
    async with async_playwright() as p:
        # Launch browser (headless=True for background, False to see it working)
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        print(f"Navigating to: {url}")
        await page.goto(url)
        
        # Wait for the page to load dynamic content.
        # We wait for a generic indicator of content, like the body or a specific div if known.
        # 'networkidle' is useful for SPAs to ensure initial data fetch is done.
        try:
            await page.wait_for_load_state('networkidle')
        except:
            print("Network idle timeout, proceeding anyway...")
            
        # Wait a bit more for rendering if needed
        await page.wait_for_timeout(3000)
        
        # Get page content
        content = await page.content()
        soup = BeautifulSoup(content, 'html.parser')
        
        await browser.close()
        
        return soup

def extract_data(soup):
    data = {}
    
    # --- Extraction Logic ---
    # Note: These selectors are based on common patterns. 
    # If LawNet changes structure, these need to be updated.
    
    # 1. Case Title
    # Often in an <h1> or a specific class like .case-title
    title_tag = soup.find('h1') or soup.find(class_='caseTitle') or soup.find(class_='title')
    data['Case Name'] = title_tag.get_text(strip=True) if title_tag else "Not Found"
    
    # 2. Citation
    # Often looks like "[2026] SGHC 21"
    # We can search for this pattern in the whole text if a specific tag is missing
    text = soup.get_text(" ", strip=True)
    citation_match = re.search(r'\[\d{4}\]\s+SGHC\s+\d+', text)
    data['Citation'] = citation_match.group(0) if citation_match else "Not Found"
    
    # 3. Decision Date
    # Look for 'Decision Date' label
    date_label = soup.find(string=re.compile("Decision Date", re.IGNORECASE))
    if date_label:
        # usually the date is in the next sibling or parent's text
        data['Decision Date'] = date_label.find_next().get_text(strip=True) if date_label.find_next() else date_label.parent.get_text(strip=True)
    else:
        data['Decision Date'] = "Not Found"

    # 4. Coram / Judges
    coram_label = soup.find(string=re.compile("Coram|Before", re.IGNORECASE))
    if coram_label:
        data['Coram'] = coram_label.find_next().get_text(strip=True) if coram_label.find_next() else coram_label.parent.get_text(strip=True)
    else:
        data['Coram'] = "Not Found"
        
    # 5. Full Text / Judgment
    # Usually in a main container. We try to find the longest block of text or a specific ID.
    # Common candidates: #case-body, .judgment-text, or just the body if nothing else.
    judgment_div = soup.find(id='judgment') or soup.find(class_='judgment-body') or soup.find('article')
    if judgment_div:
        data['Judgment'] = judgment_div.get_text("\n", strip=True)
    else:
        # Fallback: simple text dump
        data['Judgment'] = text[:1000] + "... (truncated, check selectors)"

    return data

## Run the Scraper

In [ ]:
url = "https://www.lawnet.com/openlaw/cases/citation/[2026]+SGHC+21?ref=sg-sc"
soup = await scrape_lawnet_case(url)
case_data = extract_data(soup)

# Display Data
for key, value in case_data.items():
    print(f"--- {key} ---")
    print(value[:500] if isinstance(value, str) else value)
    print("\n")

## Convert to DataFrame
Organize the data into a pandas DataFrame for further analysis or export.

In [ ]:
df = pd.DataFrame([case_data])
df.head()